In [ ]:
# Run this cell after Factory Reset Runtime
!pip install --upgrade pip -q

# Install core requirements, pinning problematic dependencies
!pip install -upgrade \
    "protobuf<6.0" \
    torch \
    transformers \
    accelerate \
    datasets \
    huggingface_hub \
    sentencepiece \
    pymongo \
    fastapi \
    "uvicorn[standard]" \
    pydantic \
    python-dotenv \
    "passlib[bcrypt]==4.0.1" \
    "python-jose[cryptography]" \
    sentence-transformers \
    openai

print("--- Necessary libraries installed with constraints ---")
!pip show protobuf torch transformers

from huggingface_hub import login

from google.colab import userdata
userdata.get('hf_secret')
login(token=userdata.get('hf_secret'))


In [ ]:
!pip install transformers datasets accelerate evaluate rouge-score sentencepiece -q

import torch
import json
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import os

# --- Configuration ---
BASE_MODEL_ID = "google/flan-t5-base"
DATA_FILE = "dataset_intent.jsonl"
OUTPUT_DIR = "./flan-t5-nlu-finetuned" # Where to save the fine-tuned model
MAX_INPUT_LENGTH = 128   # Max length for user input tokens
MAX_TARGET_LENGTH = 64   # Max length for target JSON output tokens
PREFIX = "Classify intent and extract topic: " # Flan-T5 prefix
HUB_MODEL_ID = "JnsDev/flan-t5-chatbot-intent"

# Training Hyperparameters
LEARNING_RATE = 1e-5
EPOCHS = 10 # Starting point
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 1
WARMUP_STEPS = 50
LOGGING_STEPS = 50
MAX_GRAD_NORM = 1.0

# --- GPU Check ---
if torch.cuda.is_available():
    print("GPU detected. Using CUDA.")
    device = torch.device("cuda")
else:
    print("No GPU detected. Using CPU (training will be very slow).")
    device = torch.device("cpu")

# --- 1. Load Model and Tokenizer ---
print(f"Loading base model and tokenizer: {BASE_MODEL_ID}")
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_ID).to(device)
except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    exit()

# --- 2. Load and Prepare Dataset ---
print(f"Loading dataset from: {DATA_FILE}")
if not os.path.exists(DATA_FILE):
     print(f"Error: Data file not found at {DATA_FILE}. Please upload it or check the path.")
     exit()

try:
    raw_dataset = load_dataset('json', data_files={'train': DATA_FILE})['train']
    print(f"Total examples loaded: {len(raw_dataset)}")
    if len(raw_dataset) == 0:
        print("Error: Loaded dataset is empty.")
        exit()

    split_datasets = raw_dataset.train_test_split(test_size=0.1, seed=42)
    print(f"Split into {len(split_datasets['train'])} train and {len(split_datasets['test'])} test examples.")

except Exception as e:
    print(f"Error loading or splitting dataset: {e}")
    exit()

# --- 3. Preprocessing Function ---
def preprocess_function(examples):
    inputs = [PREFIX + text for text in examples['input_text']]
    targets = [json_str for json_str in examples['target_json']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=MAX_TARGET_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing datasets with simplified target...")
try:
    tokenized_splits = split_datasets.map(
        preprocess_function,
        batched=True,
        remove_columns=split_datasets['train'].column_names # Remove original text columns
    )
    print("Tokenization complete.")

    print("\n--- Inspecting first tokenized example (Train) ---")
    example = tokenized_splits['train'][0]
    print(f"Input IDs: {example['input_ids']}")
    print(f"Decoded Input: '{tokenizer.decode(example['input_ids'])}'")
    print(f"Label IDs: {example['labels']}")
    label_ids_for_decode = [label if label != -100 else tokenizer.pad_token_id for label in example['labels']]
    print(f"Decoded Labels: '{tokenizer.decode(label_ids_for_decode)}'")
    print("--- End Inspection ---\n")

except Exception as e:
    print(f"Error during tokenization: {e}")
    exit()

# --- 4. Training Arguments ---
print("Configuring training arguments...")
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type='linear',
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=MAX_GRAD_NORM,

    save_total_limit=2,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,

    predict_with_generate=True,

    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    fp16=False,

    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_strategy="end",
    report_to="tensorboard",
)


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

# --- 7. Initialize Trainer ---
print("Initializing Trainer...")
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_splits['train'],
    eval_dataset=tokenized_splits['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# --- 8. Train ---
print("Starting training...")
try:
    train_result = trainer.train()
    print("Training finished.")

    # --- 9. Save Final Model, Metrics, State ---
    print(f"Saving best fine-tuned model to {OUTPUT_DIR}")
    trainer.save_model() # Saves the best model checkpoint based on `metric_for_best_model`

    print("Saving training metrics and state...")
    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)
    trainer.save_state()
    print("Metrics and state saved.")

    # --- 10. Evaluate Final Model ---
    print("Evaluating final model on the test set...")
    eval_metrics = trainer.evaluate()
    print("Evaluation metrics:", eval_metrics)
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)


    # --- 11. Push to Hub---
    if training_args.push_to_hub:
        print(f"Pushing model and tokenizer to Hugging Face Hub: {HUB_MODEL_ID}")
        try:
            trainer.push_to_hub(commit_message="End of training")
            print("Model pushed to Hub successfully.")
        except Exception as e:
            print(f"Error pushing model to Hub: {e}")
            print("Please ensure you are logged in (`huggingface-cli login`) and the repository exists or can be created.")

    print("\nFine-tuning complete. Model saved locally and potentially pushed to Hub.")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")

# --- 12. Inference Example ---
from transformers import pipeline

# # Load the fine-tuned model for inference
pipe = pipeline("text2text-generation", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, device=0 if torch.cuda.is_available() else -1)

test_input = "Summarize the concept of Big O notation"
query = PREFIX + test_input
result = pipe(query, max_length=MAX_TARGET_LENGTH)
print(f"Input: {test_input}")
print(f"Generated Output: {result[0]['generated_text']}")

test_input_2 = "quiz me on python lists"
query_2 = PREFIX + test_input_2
result_2 = pipe(query_2, max_length=MAX_TARGET_LENGTH)
print(f"\nInput: {test_input_2}")
print(f"Generated Output: {result_2[0]['generated_text']}")